# Programming Assignment 7 (80 points)
## Instructions
* ## Please go over materials from [lecture 22](https://colab.research.google.com/drive/1pEXZUHAJQ7KGT9XBBJz-JTUjzWuRT7fw?usp=sharing), [Lecture 23](https://colab.research.google.com/drive/1PyCEXwGoRC2gagfgtp6lPw_sGeASfYHI?usp=sharing) before starting the first 3 programming challenges in this assignment.
* ## Please go over materials from week 10 for completing programming challenge 4 and 5.

# Part 1: Scheduling with dependencies
* ## In event based simulations, we modeled the problem of scheduling events occuring in a sequence.
* ## With graphs, we explored a non-linear data structure where vertices are connected to other vertices via edges. There isn't a single sequence of data in this case e.g. vertices a, b, c, d can have edges between a and d.
* ## Both of these implementations become relevant for scheduling with dependencies.

# Class design
* ## We want a ***scheduler object which tells us the order*** in which we need to carry out tasks.
* ## It should support the following operations:
  * ## add_task: We should be able to ***add a task along with its dependencies***.
  * ## reset: We should be able to run a sequence of tasks again.
  * ## available_tasks: This property should tell us which ***tasks we can run in parallel***.
  * ## mark_completed: Notify the scheduler when a task has been completed. This should return the set of new tasks that we can run once we are done with the current task. This can happen in parallel.
  * ## all_done: returns a boolean which tells us whether all the tasks have been completed or not.     

* ## We can now design a `DependencyScheduler` class which can do the following:

## We can create a scheduler with no tasks

```
ds = DependencyScheduler()
```
## Then we want to **add** the tasks and dependencies

```
ds.add_task(a, [b, c])
ds.add_task(b, [c, d])
```
## Once the tasks are added, let's begin

```
ds.reset()
```
## We can ask for the set of tasks that we can do:
```
ds.available_tasks
```
## We want to say that we've completed the task and get the set of new tasks that we can do and so on:
```
ds.mark_completed(b)
```
## Then we want to check when we've done all tasks:
```
ds.done
```









# Connecting with the `Graph` class design
* ## This implementation will be similar, with the ***tasks corresponding to the vertices***.
* ## The main difference is that given a vertex $v$, ***we should be able to access both the predecessors (prerequisite tasks for $v$) and successors $u$ for which $v$ was declared as the predecessor***
* ## We will make use of `defaultdict` where if a key is not defined, it returns a default value.   

In [ ]:
from collections import defaultdict
import networkx as nx
import matplotlib.pyplot as plt
import random

In [ ]:
class DependencyScheduler(object):
  def __init__(self):
    self.tasks = set()
    self.predecessors = defaultdict(set) # Can only be defined once a task is completed.
    self.successors = defaultdict(set)
    self.completed_tasks = set()

  def add_task(self, t, dependencies):
    assert t not in self.tasks or len(self.predecessors[t])==0, "The task is already present"
    self.tasks.add(t)
    self.tasks.update(dependencies)
    self.predecessors[t] = set(dependencies)
    for u in dependencies:
      self.successors[u].add(t)

  def reset(self):
    self.completed_tasks = set()

  @property
  def done(self):
    return self.completed_tasks == self.tasks

  def show(self):
    g = nx.DiGraph()
    g.add_nodes_from(self.tasks)
    g.add_edges_from([(u, v) for u in self.tasks for v in self.successors[u]])
    node_colors = [('green' if v in self.completed_tasks
                    else 'red' if v in self.available_tasks
                    else 'gray')
                    for v in self.tasks]
    nx.draw(g, with_labels=True, node_color=node_colors)
    plt.show()

  @property
  def uncompleted(self):
    return self.tasks - self.completed_tasks

  @property
  def available_tasks(self):
    """
    Returns the set of tasks that can be done in parallel.
    A task can be done if all its predecessors have been completed.
    And of course, we don't return any task that has already been
    completed.
    """
    return ({t for t in self.tasks if self.predecessors[t].issubset(self.completed_tasks)}
            - self.completed_tasks)
    return set()

  def mark_completed(self, t):
    """
    Marks the task t as completed, and returns the additional
    set of tasks that can be done (and that could not be
    previously done) once t is completed.
    """
    self.completed_tasks.add(t)
    return {u for u in self.successors[t] if self.predecessors[u].issubset(self.completed_tasks)}

  def _check(self):
    """
    We want to check if t is a successor of u, then u is predecessor.
    """
    for u in self.tasks:
      for t in self.successors[u]:
        assert u in self.predecessors[t]

In [ ]:
# Code for testing -
s = DependencyScheduler()
s.add_task('a', ['b', 'c'])
s.add_task('b', ['c', 'e'])
s._check()
s.show()

In [ ]:
# Complex tests
s = DependencyScheduler()
s.add_task('a', ['b'])
s.add_task('b', ['c'])
s.add_task('c', ['a'])
assert s.available_tasks == set()
s = DependencyScheduler()
s.add_task('a', ['b'])
s.add_task('b', ['c'])
s.add_task('c', ['a', 'e'])
assert s.available_tasks == {'e'}

In [ ]:
def execute_schedule(s, show=False):
    s.reset()
    in_process = s.available_tasks
    print("Starting by doing:", in_process)
    while len(in_process) > 0:
        # Picks one random task to be the first to be completed.
        t = random.choice(list(in_process))
        print("Completed:", t)
        in_process = in_process - {t} | s.mark_completed(t)
        print("Now doing:", in_process)
        if show:
            s.show()
    # Have we done all?
    if not s.done:
        print("Error, there are tasks that could not be completed:", s.uncompleted)

In [ ]:
s = DependencyScheduler()
s.add_task('a', ['b', 'c'])
s.add_task('b', ['c', 'e'])
s._check()
s.show()

In [ ]:
execute_schedule(s, show=True)

Starting by doing: {'e', 'c'}
Completed: c
Now doing: {'e'}
Completed: e
Now doing: {'b'}
Completed: b
Now doing: {'a'}
Completed: a
Now doing: set()

# Programming Challenge 1 (10 points)
* ## (From lecture 22) First, we will create a class `RunSchedule`. With this class, we will be able to specify as parameters, the *execution policy*, what *interrupts the schedule*, and how *to resume the schedule*.
* ## Now, an object of `RunSchedule` is created alongside an object of `DependencyScheduler`
* ## It has the following methods -
  * ## `reset`: marks ***all*** tasks as ***not completed***.
  * ## `step`: performs ***one step*** in the schedule, completing a ***single task***. ***Returns the task*** that was completed.
  * ## `run`: performs ***all steps*** in the schedule until completion. ***Returns all tasks*** in the order in which they were completed.
  * ## `done`: indicates ***all tasks*** have been done.     

* ## The purpose of defining two separate classes is to ensure separation of concerns:
  * ## The scheduler is concerned with ***what can be done*** next.
  * ## The runner is concerned with the practical constraints of the execution. It is concerned with what among the possible tasks ***is actually done***.   

In [ ]:
class RunSchedule(object):

    def __init__(self, scheduler):
        self.scheduler = scheduler
        self.in_process = None # Indicating, we don't know yet.

    def reset(self):
        self.scheduler.reset()
        self.in_process = None

    def step(self):
        """Performs a step, returning the task, if any, or None,
        if there is no step that can be done."""
        # If we don't know what steps are in process, we get them.
        if self.in_process is None:
            self.in_process = self.scheduler.available_tasks
        if len(self.in_process) == 0:
            return None
        t = random.choice(list(self.in_process))
        self.in_process = self.in_process - {t} | self.scheduler.mark_completed(t)
        return t

    @property
    def done(self):
        return self.scheduler.done

    def run(self):
        """Runs the scheduler from the current configuration to completion.
        You must call reset() first, if you want to run the whole schedule."""
        tasks = []
        while not self.done:
            t = self.step()
            if t is not None:
                tasks.append(t)
        return tasks

* ## Requirements:
  * ## Imagine that you have some tasks in an intermediate step with some being completed and some not. You now want to *redo* a task by marking it as incomplete.
  * ## As a consequence, what is the set of tasks that now need to be marked as incomplete (or (hint) `undone`)? For example, you have two tasks $x$ and $y$ where $y$ is the successor of $x$. If $x$ is marked incomplete, $y$ will also need to be marked as incomplete since it might depend on the results of $x$.
  * ## We will now need to implement a `redo` method which marks a task and all of its successors (and successors of successors and so on) to be redone - ***unmarks them as `done`***.
  * ## Refer to the graph reachability implementation to find the chain of successors.
  * ## Please review the method docstring below as well.   


In [ ]:
def dependency_scheduler_redo(self, t):
    """Mark the task t, and all its successors, as undone.
    Returns the set of successor tasks of t, with t included."""
    to_redo = set()
    stack = [t]

    while stack:
        curr = stack.pop()
        if curr not in to_redo:
            to_redo.add(curr)
            stack.extend(self.successors[curr])

    self.completed_tasks -= to_redo
    return to_redo

DependencyScheduler.redo = dependency_scheduler_redo

In [ ]:
# Tests 10 points: `redo`

s = DependencyScheduler()
s.add_task('a', [])
s.add_task('b', ['a'])
s.add_task('c', ['a'])
s.add_task('d', ['b', 'c'])
s.add_task('e', ['a', 'd'])

s.mark_completed('a')
s.mark_completed('b')
s.mark_completed('c')
assert s.available_tasks == {'d'}
s.redo('b')
assert s.available_tasks == {'b'}


Tests passed, you earned 10/10 points

* ## We can now define the corresponding runner.

In [ ]:
def run_schedule_redo(self, t):
    """Marks t as to be redone."""
    # We drop everything that was in progress.
    # This also forces us to ask the scheduler for what to redo.
    self.in_process = None
    return self.scheduler.redo(t)

RunSchedule.redo = run_schedule_redo

# Programming Challenge 2 (20 points)
* ## For this challenge, we will apply the dependency scheduler to a pasta cooking recipe as follows:

In [ ]:
carbonara = DependencyScheduler()

# First, the part about cooking the pancetta.
carbonara.add_task('dice onions', [])
carbonara.add_task('dice pancetta', [])
carbonara.add_task('put oil and butter in pan', [])
carbonara.add_task('put pancetta in pan', ['dice pancetta'])
carbonara.add_task('put onions in pan', ['dice onions'])
carbonara.add_task('cook pancetta', ['put oil and butter in pan',
                                     'put pancetta in pan',
                                     'put onions in pan'])

# Second, the part about beating the eggs.
carbonara.add_task('put eggs in bowl', [])
carbonara.add_task('beat eggs', ['put eggs in bowl'])

# Third, cooking the pasta.
carbonara.add_task('fill pot with water', [])
carbonara.add_task('bring pot of water to a boil', ['fill pot with water'])
carbonara.add_task('add salt to water', ['bring pot of water to a boil'])
carbonara.add_task('put pasta in water', ['bring pot of water to a boil',
                                         'add salt to water'])
carbonara.add_task('colander pasta', ['put pasta in water'])

# And finally, we can put everything together.
carbonara.add_task('serve', ['beat eggs', 'cook pancetta', 'colander pasta'])

# Let's look at our schedule!
carbonara.show()

In [ ]:
# And let's finally prepare carbonara!
execute_schedule(carbonara)

Starting by doing: {'fill pot with water', 'put eggs in bowl', 'dice pancetta', 'put oil and butter in pan', 'dice onions'}
Completed: dice pancetta
Now doing: {'fill pot with water', 'put eggs in bowl', 'put oil and butter in pan', 'put pancetta in pan', 'dice onions'}
Completed: dice onions
Now doing: {'fill pot with water', 'put eggs in bowl', 'put oil and butter in pan', 'put onions in pan', 'put pancetta in pan'}
Completed: put pancetta in pan
Now doing: {'fill pot with water', 'put eggs in bowl', 'put oil and butter in pan', 'put onions in pan'}
Completed: put onions in pan
Now doing: {'fill pot with water', 'put eggs in bowl', 'put oil and butter in pan'}
Completed: fill pot with water
Now doing: {'put oil and butter in pan', 'bring pot of water to a boil', 'put eggs in bowl'}
Completed: put eggs in bowl
Now doing: {'put oil and butter in pan', 'bring pot of water to a boil', 'beat eggs'}
Completed: beat eggs
Now doing: {'put oil and butter in pan', 'bring pot of water to a boil

* ## For this challenge, we will implement `redo` for the cooking tasks.
* ## In code, when one file changes, you only need to redo the things that depend on that file. In cooking, it is different: even if nothing changed in the bacon, onions, and cooked pasta, once you add to it rotten eggs you have to redo the pasta, bacon, onions, etc, as well, as they have now been contaminated.
* ## The root of the problem is that in a makefile, when you combine two files to compute a result, you do not destroy the original files, whereas in cooking, once you combine foods, you don't have the original foods any longer. Cooking is like a makefile in which, once you combine files, you immediately delete them.


* ## We need to come up with a precise definition of what needs to be redone. Similar to the previous challenge, we first label a task that needs to be redone. Then we propogate this label as follows:
  * ## **Forward**: Apply the following rule (as long as possible): Consider a task $v$ labeled *redo*. If ***$u$ is a successor for $v$*** and $u$ has been completed, then $u$ will also be labeled *redo*.
  * ## **Backward**: Apply the following rule (as long as possible): Consider a task $v$ labeled *redo*. If ***$u$ is a predecessor of $v$***, then $u$ is labeled *redo*. Note that here, $u$ is guaranteed to have been completed.
  * ## Note that you have to implement once the forward phase, followed by once the backward phase. Example: you press some lemons, and you use a bit of the juice to make a mayonnaise, and another part to marinate shrimp. If it turns out that the egg yolk used for the mayonnaise was rotten, you need to press a bit of lemon again, but you do not need to marinate the shrimp once more.    

In [ ]:
def dependency_scheduler_cooking_redo(self, v):
    """Indicates that the task v needs to be redone, as something went bad.
    This is the "cooking" version of the redo, in which the redo propagates
    to both successors (as for code) and predecessors."""
    redo_set = {v}

    # Forward phase: redo completed successors
    stack = [v]
    while stack:
        curr = stack.pop()
        for succ in self.successors[curr]:
            if succ in self.completed_tasks and succ not in redo_set:
                redo_set.add(succ)
                stack.append(succ)

    # Backward phase: redo all predecessors of anything marked redo
    stack = list(redo_set)
    while stack:
        curr = stack.pop()
        for pred in self.predecessors[curr]:
            if pred not in redo_set:
                redo_set.add(pred)
                stack.append(pred)

    self.completed_tasks -= redo_set
    return redo_set

DependencyScheduler.cooking_redo = dependency_scheduler_cooking_redo

In [ ]:
# Tests 10 points: Basic tests for `cooking_redo`

s = DependencyScheduler()
s.add_task('a', [])
s.add_task('b', [])
s.add_task('c', ['a', 'b'])
s.add_task('d', ['c', 'a'])
s.add_task('e', [])
s.add_task('f', ['e'])
s.add_task('g', ['f', 'd'])

s.mark_completed('a')
s.mark_completed('b')
s.mark_completed('c')
s.mark_completed('d')
assert s.available_tasks == {'e'}
s.cooking_redo('c')
# When we redo c, both its successor d, and predecessors a, b have to be redone.
assert s.available_tasks == {'a', 'b', 'e'}
assert s.completed_tasks == set()


Tests passed, you earned 10/10 points

In [ ]:
# Tests 10 points: Advanced tests for `cooking_redo`

s = DependencyScheduler()
s.add_task('a', [])
s.add_task('b', [])
s.add_task('c', ['a', 'b'])
s.add_task('d', ['c', 'a'])
s.add_task('e', [])
s.add_task('f', ['e'])
s.add_task('g', ['f', 'd'])

s.mark_completed('a')
s.mark_completed('b')
s.mark_completed('c')
s.mark_completed('d')
s.mark_completed('e')
assert s.available_tasks == {'f'}
s.cooking_redo('c')
# When we redo c, both its successor d, and predecessors a, b have to be redone.
assert s.available_tasks == {'a', 'b', 'f'}
assert s.completed_tasks == {'e'}


Tests passed, you earned 10/10 points

# Programming Challenge 3 (30 points)
* ## In the schedules we have seen so far, the dependencies are in an `and` relation with one another; If task $a$ depends on $[b, c]$, then ***both $b$ and $c$*** need to be completed before $a$ can be completed.
* ## It is possible that the dependencies are in an `or` relation with one another; If task $a$ depends on $[b, c]$, then ***either $b$ or $c$*** need to be completed before $a$ can be complete.
* ## We can apply this idea to the Carbonara Pasta example as follows:
  * ## We can add shallots instead of the onions. In this case, the statement:
  ```
  carbonara.add_task('put onions in pan', ['dice onions'])  
  ```
  ## can be replaced with
  ```
  carbonara.add_or_task('put onions in pan', ['dice onions', 'dice shallots'])
  ```
* ## We will need to add the following methods to the `Scheduler` class:
  * ## `add_or_task(self, t, dependencies)`: adds a task `t` with list of dependencies `dependencies`, so that `t` can be done when all of the dependencies are done. The task `t` is called an AND node in the dependency graph.
  * ## `add_or_task(self, t, dependencies)` adds a task `t` with list of dependencies dependencies, so that `t` can be done when at least one of the dependencies is done. The task `t` is called an OR node in the dependency graph
  * ## There will need to be a way of separately storing the `and` and `or` tasks. In addition, you will also need to implement the properties `done`, `available_tasks`, `uncompleted` and the method `mark_completed`. The `show()` method implementation is optional but can be useful for debugging.   


    

In [ ]:
class AND_OR_Scheduler(object):

    def __init__(self):
        self.tasks = set()
        self.predecessors = defaultdict(set)
        self.successors = defaultdict(set)
        self.completed_tasks = set()
        self.and_tasks = set()
        self.or_tasks = set()

    def add_and_task(self, t, dependencies):
        """Adds an AND task t with given dependencies."""
        self.tasks.add(t)
        self.tasks.update(dependencies)
        self.predecessors[t] = set(dependencies)
        self.and_tasks.add(t)
        self.or_tasks.discard(t)
        for u in dependencies:
            self.successors[u].add(t)

    def add_or_task(self, t, dependencies):
        """Adds an OR task t with given dependencies."""
        self.tasks.add(t)
        self.tasks.update(dependencies)
        self.predecessors[t] = set(dependencies)
        self.or_tasks.add(t)
        self.and_tasks.discard(t)
        for u in dependencies:
            self.successors[u].add(t)

    @property
    def done(self):
        return self.completed_tasks == self.tasks

    @property
    def available_tasks(self):
        """Returns the set of tasks that can be done in parallel.
        A task can be done if:
        - It is an AND task, and all its predecessors have been completed, or
        - It is an OR task, and at least one of its predecessors has been completed.
        And of course, we don't return any task that has already been
        completed."""
        available = set()

        for t in self.tasks:
            if t in self.completed_tasks:
                continue

            preds = self.predecessors[t]

            # Base tasks with no predecessors are available
            if len(preds) == 0:
                available.add(t)
            elif t in self.and_tasks and preds.issubset(self.completed_tasks):
                available.add(t)
            elif t in self.or_tasks and len(preds & self.completed_tasks) > 0:
                available.add(t)

        return available

    def mark_completed(self, t):
        """Marks the task t as completed, and returns the additional
        set of tasks that can be done (and that could not be
        previously done) once t is completed."""
        self.completed_tasks.add(t)
        newly_available = set()

        for u in self.successors[t]:
            if u in self.completed_tasks:
                continue
            preds = self.predecessors[u]
            if u in self.and_tasks and preds.issubset(self.completed_tasks):
                newly_available.add(u)
            elif u in self.or_tasks and len(preds & self.completed_tasks) > 0:
                newly_available.add(u)

        return newly_available

    def show(self):
        """You can use the nx graph to display the graph.  You may want to ensure
        that you display AND and OR nodes differently."""
        g = nx.DiGraph()
        g.add_nodes_from(self.tasks)
        g.add_edges_from([(u, v) for u in self.tasks for v in self.successors[u]])

        node_colors = []
        for v in self.tasks:
            if v in self.completed_tasks:
                node_colors.append('green')
            elif v in self.available_tasks:
                node_colors.append('red')
            elif v in self.or_tasks:
                node_colors.append('orange')
            else:
                node_colors.append('lightblue')

        nx.draw(g, with_labels=True, node_color=node_colors)
        plt.show()

* ## Below are the tests for the ***AND*** nodes.

In [ ]:
# Tests 10 points: Simple tests for AND nodes.

s = AND_OR_Scheduler()
s.add_and_task('a', ['b', 'c'])
assert s.available_tasks == {'b', 'c'}
r = s.mark_completed('b')
assert r == set()
assert s.available_tasks == {'c'}
r = s.mark_completed('c')
assert r == {'a'}
assert s.available_tasks == {'a'}
r = s.mark_completed('a')
assert r == set()
assert s.available_tasks == set()


Tests passed, you earned 10/10 points

* ## Below are the tests for the ***OR*** nodes.

In [ ]:
# Tests 10 points: Simple tests for OR nodes.

s = AND_OR_Scheduler()
s.add_or_task('a', ['b', 'c'])
assert s.available_tasks == {'b', 'c'}
r = s.mark_completed('b')
# Now 'a' becomes available.
assert r == {'a'}
# But note that 'c' is also available, even if useless.
assert s.available_tasks == {'a', 'c'}
r = s.mark_completed('a')
assert r == set()
assert s.available_tasks == {'c'}
r = s.mark_completed('c')
assert r == set()
assert s.available_tasks == set()


Tests passed, you earned 10/10 points

In [ ]:
# Tests 10 points: Tests with both AND and OR nodes.

s = AND_OR_Scheduler()
s.add_and_task('a', ['b', 'c'])
s.add_or_task('b', ['b1', 'b2'])
s.add_or_task('c', ['c1', 'c2'])
r = s.mark_completed('b1')
assert s.available_tasks == {'b', 'b2', 'c1', 'c2'}
r = s.mark_completed('b')
assert 'a' not in s.available_tasks
r = s.mark_completed('c1')
assert 'a' not in s.available_tasks
r = s.mark_completed('c')
assert 'a' in s.available_tasks

s = AND_OR_Scheduler()
s.add_or_task('a', ['b', 'c'])
s.add_and_task('b', ['b1', 'b2'])
s.add_and_task('c', ['c1', 'c2'])
r = s.mark_completed('b1')
assert s.available_tasks == {'b2', 'c1', 'c2'}
r = s.mark_completed('c1')
assert s.available_tasks == {'b2', 'c2'}
r = s.mark_completed('c2')
assert s.available_tasks == {'b2', 'c'}
r = s.mark_completed('c')
assert 'a' in s.available_tasks


Tests passed, you earned 10/10 points

# Part 2: Writing a Sudoku solver - 20 points
* ## Here, we will write a [Sudoku](https://en.wikipedia.org/wiki/Sudoku) solver.  We want to get as input a Sudoku with some cells filled with values, and we want to get as output a solution, if one exists, and otherwise a notice that the input Sudoku puzzle has no solutions.
* ## The way we go about solving Sudoku is prototypical of a very large number of problems in computer science.  In these problems, the solution is attained through a mix of search (we attempt to fill a square with a number and see if it works out), and constraint propagation (if we fill a square with, say, a 1, then there can be no 1's in the same row, column, and 3x3 square).

* ## One initial idea would be to represent a Sudoku problem via a $9 \times 9$ matrix, where each entry can be either a digit from 1 to 9, or 0 to signify "blank".  This would work in some sense, but it would not be a very useful representation.  If you have solved Sudoku by hand (and if you have not, please go and solve a couple; it will teach you a lot about what we need to do), you will know that the following strategy works:
  * ## Repeat the following:
    * ## Look at all blank spaces.  Can you find one where only one digit fits? If so, write the digit there.
    * ## If you cannot find any blank space as above, try to find one where only a couple or so digits can fit.  Try putting in one of those digits, and see if you can solve the puzzle with that choice.  If not, backtrack, and try another digit.

* ## Thus, it will be very useful to us to remember not only the known digits, but also, which digits can fit into any blank space.
  * ## Hence, we represent a Sudoku problem via a $9 \times 9$ matrix of _sets_: each set contains the digits that can fit in a given space.
  * ## Of course, a known digit is just a set containing only one element.
* ## We will solve a Sudoku problem by progressively "shrinking" these sets of possibilities, until they all contain exactly one element.

* ## First, let us write some code that enables us to define a Sudoku problem, and display it for us.

In [ ]:
import json
from collections import defaultdict


In [ ]:
def getel(s):
    """Returns the unique element in a singleton set (or list). A singleton would an object containing exactly 1 item."""
    assert len(s) == 1
    return list(s)[0]

In [ ]:
class Sudoku(object):
  def __init__(self, elements):
    """Parameter elements is an object that can be one of the following:
    Case 1: a list of 9 strings of length 9 each.
    Each string represents a row of the initial Sudoku puzzle,
    with either a digit 1..9 in it, or with a blank or _ to signify
    a blank cell.
    Case 2: an instance of Sudoku.  In that case, we initialize an
    object to be equal (a copy) of the one in elements.
    Case 3: a list of list of sets, used to initialize the problem."""
    if isinstance(elements, Sudoku):
      # We let self.m consist of copies of each set in elements.m
      self.m = [[x.copy() for x in row] for row in elements.m]
    else:
      assert len(elements) == 9
      for s in elements:
        assert len(s) == 9
      # We let self.m be our Sudoku problem, a 9x9 matrix of sets.
      self.m = []
      for s in elements:
        row = []
        for c in s:
          if isinstance(c, str):
            if c.isdigit():
              row.append({int(c)})
            else:
              row.append({1, 2, 3, 4, 5, 6, 7, 8, 9})
          else:
            assert isinstance(c, set)
            row.append(c)
        self.m.append(row)


  def show(self, details=False):
    """Prints out the Sudoku matrix.  If details=False, we print out
    the digits only for cells that have singleton sets (where only
    one digit can fit).  If details=True, for each cell, we display the
    sets associated with the cell."""
    if details:
      print("+-----------------------------+-----------------------------+-----------------------------+")
      for i in range(9):
        r = '|'
        for j in range(9):
          # We represent the set {2, 3, 5} via _23_5____
          s = ''
          for k in range(1, 10):
              s += str(k) if k in self.m[i][j] else '_'
          r += s
          r += '|' if (j + 1) % 3 == 0 else ' '
        print(r)
        if (i + 1) % 3 == 0:
          print("+-----------------------------+-----------------------------+-----------------------------+")
    else:
      print("+---+---+---+")
      for i in range(9):
        r = '|'
        for j in range(9):
          if len(self.m[i][j]) == 1:
              r += str(getel(self.m[i][j]))
          else:
              r += "."
          if (j + 1) % 3 == 0:
              r += "|"
        print(r)
        if (i + 1) % 3 == 0:
          print("+---+---+---+")


  def to_string(self):
    """This method is useful for producing a representation that
    can be used in testing."""
    as_lists = [[list(self.m[i][j]) for j in range(9)] for i in range(9)]
    return json.dumps(as_lists)


  @staticmethod
  def from_string(s):
    """Inverse of above."""
    as_lists = json.loads(s)
    as_sets = [[set(el) for el in row] for row in as_lists]
    return Sudoku(as_sets)


  def __eq__(self, other):
    """Useful for testing."""
    return self.m == other.m

* ## Let us input a problem (the Sudoku example found on [this Wikipedia page](https://en.wikipedia.org/wiki/Sudoku)) and check that our serialization and deserialization works.

In [ ]:
sd = Sudoku([
    '53__7____',
    '6__195___',
    '_98____6_',
    '8___6___3',
    '4__8_3__1',
    '7___2___6',
    '_6____28_',
    '___419__5',
    '____8__79'
])
sd.show()
sd.show(details=True)
s = sd.to_string()
sdd = Sudoku.from_string(s)
sdd.show(details=True)
assert sd == sdd

+---+---+---+
|53.|.7.|...|
|6..|195|...|
|.98|...|.6.|
+---+---+---+
|8..|.6.|..3|
|4..|8.3|..1|
|7..|.2.|..6|
+---+---+---+
|.6.|...|28.|
|...|419|..5|
|...|.8.|.79|
+---+---+---+
+-----------------------------+-----------------------------+-----------------------------+
|____5____ __3______ 123456789|123456789 ______7__ 123456789|123456789 123456789 123456789|
|_____6___ 123456789 123456789|1________ ________9 ____5____|123456789 123456789 123456789|
|123456789 ________9 _______8_|123456789 123456789 123456789|123456789 _____6___ 123456789|
+-----------------------------+-----------------------------+-----------------------------+
|_______8_ 123456789 123456789|123456789 _____6___ 123456789|123456789 123456789 __3______|
|___4_____ 123456789 123456789|_______8_ 123456789 __3______|123456789 123456789 1________|
|______7__ 123456789 123456789|123456789 _2_______ 123456789|123456789 123456789 _____6___|
+-----------------------------+-----------------------------+---------------------

## Constraint propagation

* ## When the set in a Sudoku cell contains only one element, this means that the digit at that cell is known.
* ## We can then propagate the knowledge, ruling out that digit in the same row, in the same column, and in the same 3x3 cell.
* ## We first write a method that propagates the constraint from a single cell.  The method will return the list of newly-determined cells, that is, the list of cells who also now (but not before) are associated with a 1-element set.  This is useful, because we can then propagate the constraints from those cells in turn.  Further, if an empty set is ever generated, we raise the exception Unsolvable: this means that there is no solution to the proposed Sudoku puzzle.

* ## Propagating constraints from a single cell

In [ ]:
class Unsolvable(Exception):
    pass


def sudoku_ruleout(self, i, j, x):
    """The input consists in a cell (i, j), and a value x.
    The function removes x from the set self.m[i][j] at the cell, if present, and:
    - if the result is empty, raises Unsolvable;
    - if the cell used to be a non-singleton cell and is now a singleton
      cell, then returns the set {(i, j)};
    - otherwise, returns the empty set."""
    c = self.m[i][j]
    n = len(c)
    c.discard(x)
    self.m[i][j] = c
    if len(c) == 0:
        raise Unsolvable()
    return {(i, j)} if 1 == len(c) < n else set()

Sudoku.ruleout = sudoku_ruleout

* ## Here's the method definition for sudoku cell propagation. Write code for propagating constraints from the cell for:
  * ## The same column.
  * ## The same block of 3x3 cells.

In [ ]:
def sudoku_propagate_cell(self, ij):
    """Propagates the singleton value at cell (i, j), returning the list
    of newly-singleton cells."""
    i, j = ij
    if len(self.m[i][j]) > 1:
        # Nothing to propagate from cell (i,j).
        return set()
    # We keep track of the newly-singleton cells.
    newly_singleton = set()
    x = getel(self.m[i][j]) # Value at (i, j).
    # Same row.
    for jj in range(9):
        if jj != j: # Do not propagate to the element itself.
            newly_singleton.update(self.ruleout(i, jj, x))
    # Same column.
    for ii in range(9):
        if ii != i:
            newly_singleton.update(self.ruleout(ii, j, x))
    # Same block of 3x3 cells.
    bi = (i // 3) * 3
    bj = (j // 3) * 3
    for ii in range(bi, bi + 3):
        for jj in range(bj, bj + 3):
            if (ii, jj) != (i, j):
                newly_singleton.update(self.ruleout(ii, jj, x))
    # Returns the list of newly-singleton cells.
    return newly_singleton

Sudoku.propagate_cell = sudoku_propagate_cell

In [ ]:
# Tests 10 points: cell propagation

tsd = Sudoku.from_string('[[[5], [3], [2], [6], [7], [8], [9], [1, 2, 4], [2]], [[6], [7], [1, 2, 4, 7], [1, 2, 3], [9], [5], [3], [1, 2, 4], [8]], [[1, 2], [9], [8], [3], [4], [1, 2], [5], [6], [7]], [[8], [5], [9], [1, 9, 7], [6], [1, 4, 9, 7], [4], [2], [3]], [[4], [2], [6], [8], [5], [3], [7], [9], [1]], [[7], [1], [3], [9], [2], [4], [8], [5], [6]], [[1, 9], [6], [1, 5, 9, 7], [9, 5, 7], [3], [9, 7], [2], [8], [4]], [[9, 2], [8], [9, 2, 7], [4], [1], [9, 2, 7], [6], [3], [5]], [[3], [4], [2, 3, 4, 5], [2, 5, 6], [8], [6], [1], [7], [9]]]')
tsd.show(details=True)
try:
    tsd.propagate_cell((0, 2))
except Unsolvable:
    print("Good! It was unsolvable.")
else:
    raise Exception("Hey, it was unsolvable")

tsd = Sudoku.from_string('[[[5], [3], [2], [6], [7], [8], [9], [1, 2, 4], [2, 3]], [[6], [7], [1, 2, 4, 7], [1, 2, 3], [9], [5], [3], [1, 2, 4], [8]], [[1, 2], [9], [8], [3], [4], [1, 2], [5], [6], [7]], [[8], [5], [9], [1, 9, 7], [6], [1, 4, 9, 7], [4], [2], [3]], [[4], [2], [6], [8], [5], [3], [7], [9], [1]], [[7], [1], [3], [9], [2], [4], [8], [5], [6]], [[1, 9], [6], [1, 5, 9, 7], [9, 5, 7], [3], [9, 7], [2], [8], [4]], [[9, 2], [8], [9, 2, 7], [4], [1], [9, 2, 7], [6], [3], [5]], [[3], [4], [2, 3, 4, 5], [2, 5, 6], [8], [6], [1], [7], [9]]]')
tsd.show(details=True)
assert  tsd.propagate_cell((0, 2)) == {(0, 8), (2, 0)}


Tests passed, you earned 10/10 points

+-----------------------------+-----------------------------+-----------------------------+
|____5____ __3______ _2_______|_____6___ ______7__ _______8_|________9 12_4_____ _2_______|
|_____6___ ______7__ 12_4__7__|123______ ________9 ____5____|__3______ 12_4_____ _______8_|
|12_______ ________9 _______8_|__3______ ___4_____ 12_______|____5____ _____6___ ______7__|
+-----------------------------+-----------------------------+-----------------------------+
|_______8_ ____5____ ________9|1_____7_9 _____6___ 1__4__7_9|___4_____ _2_______ __3______|
|___4_____ _2_______ _____6___|_______8_ ____5____ __3______|______7__ ________9 1________|
|______7__ 1________ __3______|________9 _2_______ ___4_____|_______8_ ____5____ _____6___|
+-----------------------------+-----------------------------+-----------------------------+
|1_______9 _____6___ 1___5_7_9|____5_7_9 __3______ ______7_9|_2_______ _______8_ ___4_____|
|_2______9 _______8_ _2____7_9|___4_____ 1________ _2____7_9|_____6___ __3______

* ## We can now propagate each cell, once.

In [ ]:
def sudoku_propagate_all_cells_once(self):
    """This function propagates the constraints from all singletons."""
    for i in range(9):
        for j in range(9):
            self.propagate_cell((i, j))

Sudoku.propagate_all_cells_once = sudoku_propagate_all_cells_once

In [ ]:
sd = Sudoku([
    '53__7____',
    '6__195___',
    '_98____6_',
    '8___6___3',
    '4__8_3__1',
    '7___2___6',
    '_6____28_',
    '___419__5',
    '____8__79'
])
sd.show()
sd.propagate_all_cells_once()
sd.show()
sd.show(details=True)

+---+---+---+
|53.|.7.|...|
|6..|195|...|
|.98|...|.6.|
+---+---+---+
|8..|.6.|..3|
|4..|8.3|..1|
|7..|.2.|..6|
+---+---+---+
|.6.|...|28.|
|...|419|..5|
|...|.8.|.79|
+---+---+---+
+---+---+---+
|53.|.7.|...|
|6..|195|...|
|.98|...|.6.|
+---+---+---+
|8..|.6.|..3|
|4..|853|..1|
|7..|.2.|..6|
+---+---+---+
|.6.|..7|284|
|...|419|.35|
|...|.8.|.79|
+---+---+---+
+-----------------------------+-----------------------------+-----------------------------+
|____5____ __3______ 12_4_____|_2___6___ ______7__ _2_4_6_8_|1__4___89 12_4____9 _2_4___8_|
|_____6___ _2_4__7__ _2_4__7__|1________ ________9 ____5____|__34__78_ _234_____ _2_4__78_|
|12_______ ________9 _______8_|_23______ __34_____ _2_4_____|1_345_7__ _____6___ _2_4__7__|
+-----------------------------+-----------------------------+-----------------------------+
|_______8_ 12__5____ 12__5___9|____5_7_9 _____6___ 1__4__7__|___45_7_9 _2_45___9 __3______|
|___4_____ _2__5____ _2__56__9|_______8_ ____5____ __3______|____5_7_9 _2__5___9 1__

### Propagating all cells, repeatedly

* ## This is a good beginning, but it's not quite enough. As we propagate the constraints, cells that did not use to be singletons may have become singletons.  For example, in the above example, the center cell has become known to be a 5: we need to make sure that also these singletons are propagated.

* ## This is why we have written `propagate_cell` so that it returns the set of newly-singleton cells.  

* ## We need now to write a method full_propagation that
  * ## At the beginning starts with a set of `to_propagate` cells (if it is not specified, then we just take it to consist of all singleton cells).  
  * ## Then, it picks a cell from the `to_propagate` set, and propagates from it, adding any newly singleton cell to `to_propagate`.  
  * ## Once there are no more cells to be propagated, the method returns.

* ## If this sounds similar to graph reachability, it is ... because it is!  It is once again the algorithmic pattern of keeping a list of work to be done, then iteratively picking an element from the list, doing it, possibly updating the list of work to be done with new work that has to be done as a result of what we just did, and continuing in this fashion until there is nothing left to do.

* ## The portion you have to write can be done in three lines of code.

In [ ]:
def sudoku_full_propagation(self, to_propagate=None):
    """Iteratively propagates from all singleton cells, and from all
    newly discovered singleton cells, until no more propagation is possible."""
    if to_propagate is None:
        to_propagate = {(i, j) for i in range(9) for j in range(9)}
    # This code is the (A) code; will be referenced later.
    while len(to_propagate) > 0:
        cell = to_propagate.pop()
        to_propagate.update(self.propagate_cell(cell))

Sudoku.full_propagation = sudoku_full_propagation

In [ ]:
# Tests 10 points: full propagation

sd = Sudoku([
    '53__7____',
    '6__195___',
    '_98____6_',
    '8___6___3',
    '4__8_3__1',
    '7___2___6',
    '_6____28_',
    '___419__5',
    '____8__79'
])
sd.full_propagation()
sd.show(details=True)
sdd = Sudoku.from_string('[[[5], [3], [4], [6], [7], [8], [9], [1], [2]], [[6], [7], [2], [1], [9], [5], [3], [4], [8]], [[1], [9], [8], [3], [4], [2], [5], [6], [7]], [[8], [5], [9], [7], [6], [1], [4], [2], [3]], [[4], [2], [6], [8], [5], [3], [7], [9], [1]], [[7], [1], [3], [9], [2], [4], [8], [5], [6]], [[9], [6], [1], [5], [3], [7], [2], [8], [4]], [[2], [8], [7], [4], [1], [9], [6], [3], [5]], [[3], [4], [5], [2], [8], [6], [1], [7], [9]]]')
assert sd == sdd


Tests passed, you earned 10/10 points

+-----------------------------+-----------------------------+-----------------------------+
|____5____ __3______ ___4_____|_____6___ ______7__ _______8_|________9 1________ _2_______|
|_____6___ ______7__ _2_______|1________ ________9 ____5____|__3______ ___4_____ _______8_|
|1________ ________9 _______8_|__3______ ___4_____ _2_______|____5____ _____6___ ______7__|
+-----------------------------+-----------------------------+-----------------------------+
|_______8_ ____5____ ________9|______7__ _____6___ 1________|___4_____ _2_______ __3______|
|___4_____ _2_______ _____6___|_______8_ ____5____ __3______|______7__ ________9 1________|
|______7__ 1________ __3______|________9 _2_______ ___4_____|_______8_ ____5____ _____6___|
+-----------------------------+-----------------------------+-----------------------------+
|________9 _____6___ 1________|____5____ __3______ ______7__|_2_______ _______8_ ___4_____|
|_2_______ _______8_ ______7__|___4_____ 1________ ________9|_____6___ __3______